# Treinamento — detecção de violência

Fluxo:

1. Baixar o dataset RLVS pelo KaggleHub.
2. Identificar vídeos de violência e não violência.
3. Separar treino, validação e teste por vídeo.
4. Extrair quadros distribuídos ao longo de cada vídeo.
5. Treinar MobileNetV2 com agregação temporal.
6. Avaliar o modelo.
7. Exportar para Keras e TensorFlow Lite.

In [ ]:
!pip -q install kagglehub

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import json
import random
import re
import unicodedata

import cv2
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tqdm.auto import tqdm

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

if not tf.config.list_physical_devices("GPU"):
    print("AVISO: Falha ao usar GPU.")

## 1. Configuração

In [ ]:
@dataclass(frozen=True)
class Configuracao:
    semente: int = 42
    tamanho_imagem: int = 160
    quantidade_quadros: int = 8
    tamanho_lote: int = 4

    epocas_cabeca: int = 8
    epocas_ajuste: int = 6

    taxa_aprendizado: float = 1e-3
    taxa_ajuste: float = 5e-5

    alfa_mobilenet: float = 0.35
    dropout: float = 0.30
    camadas_ajustaveis: int = 20
    paciencia: int = 4

    limiar: float = 0.50
    proporcao_validacao: float = 0.15
    proporcao_teste: float = 0.15


configuracao = Configuracao()

PASTA_RAIZ = Path("/content/violence-edge-ai")
PASTA_DADOS = PASTA_RAIZ / "datasets"
PASTA_MODELOS = PASTA_RAIZ / "models"
PASTA_RESULTADOS = PASTA_RAIZ / "outputs"

for pasta in (PASTA_DADOS, PASTA_MODELOS, PASTA_RESULTADOS):
    pasta.mkdir(parents=True, exist_ok=True)

random.seed(configuracao.semente)
np.random.seed(configuracao.semente)
tf.keras.utils.set_random_seed(configuracao.semente)

configuracao

## 2. Baixar o dataset RLVS

In [ ]:
caminho_dataset = kagglehub.dataset_download(
    "mohamedmustafa/real-life-violence-situations-dataset",
    output_dir=str(PASTA_DADOS),
)

caminho_dataset = Path(caminho_dataset)
print("Dataset:", caminho_dataset)

## 3. Localizar e classificar os vídeos

In [ ]:
EXTENSOES_VIDEO = {".avi", ".mp4", ".mov", ".mkv", ".webm"}

NOMES_VIOLENCIA = {
    "violence",
    "violent",
    "fight",
    "fights",
}

NOMES_NAO_VIOLENCIA = {
    "nonviolence",
    "nonviolent",
    "nonfight",
    "normal",
    "naoviolencia",
}


def normalizar_nome(texto: str) -> str:
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(
        caractere
        for caractere in texto
        if not unicodedata.combining(caractere)
    )
    return re.sub(r"[^a-z0-9]", "", texto.lower())


def inferir_classe(caminho: Path):
    partes = [normalizar_nome(parte) for parte in caminho.parts]

    for parte in reversed(partes):
        if parte in NOMES_NAO_VIOLENCIA:
            return 0, "nao_violencia"

        if parte in NOMES_VIOLENCIA:
            return 1, "violencia"

    return None


def video_valido(caminho: Path) -> bool:
    captura = cv2.VideoCapture(str(caminho))
    abriu = captura.isOpened()
    quantidade_quadros = int(captura.get(cv2.CAP_PROP_FRAME_COUNT))
    captura.release()

    return abriu and quantidade_quadros > 0


def listar_videos(pasta_dataset: Path) -> pd.DataFrame:
    registros = []

    caminhos = sorted(pasta_dataset.rglob("*"))

    for caminho in tqdm(
        caminhos,
        desc="Verificando vídeos",
        unit="arquivo",
    ):
        if caminho.suffix.lower() not in EXTENSOES_VIDEO:
            continue

        classe = inferir_classe(caminho)

        if classe is None or not video_valido(caminho):
            continue

        rotulo, nome_classe = classe

        registros.append(
            {
                "caminho": str(caminho.resolve()),
                "rotulo": rotulo,
                "classe": nome_classe,
            }
        )

    tabela = pd.DataFrame(registros)

    if tabela.empty:
        raise RuntimeError(
            "Nenhum vídeo foi reconhecido. "
            "Verifique a estrutura do dataset baixado."
        )

    if tabela["rotulo"].nunique() != 2:
        raise RuntimeError("As duas classes não foram encontradas.")

    return tabela


tabela_videos = listar_videos(caminho_dataset)

print("Total de vídeos:", len(tabela_videos))
display(tabela_videos["classe"].value_counts().rename("quantidade"))
display(tabela_videos.head())

## 4. Dividir treino, validação e teste

In [ ]:
treino_validacao, teste = train_test_split(
    tabela_videos,
    test_size=configuracao.proporcao_teste,
    random_state=configuracao.semente,
    stratify=tabela_videos["rotulo"],
)

proporcao_validacao_relativa = (
    configuracao.proporcao_validacao
    / (1.0 - configuracao.proporcao_teste)
)

treino, validacao = train_test_split(
    treino_validacao,
    test_size=proporcao_validacao_relativa,
    random_state=configuracao.semente,
    stratify=treino_validacao["rotulo"],
)

treino = treino.reset_index(drop=True)
validacao = validacao.reset_index(drop=True)
teste = teste.reset_index(drop=True)

for nome, tabela in (
    ("Treino", treino),
    ("Validação", validacao),
    ("Teste", teste),
):
    print(f"\n{nome}: {len(tabela)} vídeos")
    print(tabela["classe"].value_counts())

## 5. Leitura e preparação dos vídeos

In [ ]:
def redimensionar_com_corte_central(
    quadro: np.ndarray,
    tamanho: int,
) -> np.ndarray:
    altura, largura = quadro.shape[:2]
    escala = max(tamanho / largura, tamanho / altura)

    nova_largura = max(tamanho, int(round(largura * escala)))
    nova_altura = max(tamanho, int(round(altura * escala)))

    quadro = cv2.resize(
        quadro,
        (nova_largura, nova_altura),
        interpolation=cv2.INTER_AREA,
    )

    inicio_x = (nova_largura - tamanho) // 2
    inicio_y = (nova_altura - tamanho) // 2

    return quadro[
        inicio_y : inicio_y + tamanho,
        inicio_x : inicio_x + tamanho,
    ]


def extrair_quadros(
    caminho_video: str,
    quantidade_quadros: int,
    tamanho_imagem: int,
) -> np.ndarray:
    captura = cv2.VideoCapture(caminho_video)

    if not captura.isOpened():
        raise ValueError(f"Não foi possível abrir: {caminho_video}")

    total_quadros = int(captura.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_quadros <= 0:
        captura.release()
        raise ValueError(f"Vídeo sem quadros válidos: {caminho_video}")

    indices = np.linspace(
        0,
        total_quadros - 1,
        quantidade_quadros,
        dtype=int,
    )

    quadros = []

    for indice in indices:
        captura.set(cv2.CAP_PROP_POS_FRAMES, int(indice))
        sucesso, quadro = captura.read()

        if not sucesso:
            if quadros:
                quadros.append(quadros[-1].copy())
                continue

            captura.release()
            raise ValueError(f"Falha ao ler: {caminho_video}")

        quadro = cv2.cvtColor(quadro, cv2.COLOR_BGR2RGB)
        quadro = redimensionar_com_corte_central(
            quadro,
            tamanho_imagem,
        )
        quadros.append(quadro)

    captura.release()

    return np.asarray(quadros, dtype=np.float32)


def aumentar_video(video: np.ndarray) -> np.ndarray:
    if np.random.random() < 0.5:
        video = video[:, :, ::-1, :]

    brilho = np.random.uniform(0.85, 1.15)
    return np.clip(video * brilho, 0, 255)


class GeradorVideos(keras.utils.Sequence):
    def __init__(
        self,
        tabela: pd.DataFrame,
        configuracao: Configuracao,
        aumentar: bool,
        embaralhar: bool,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.tabela = tabela.reset_index(drop=True)
        self.configuracao = configuracao
        self.aumentar = aumentar
        self.embaralhar = embaralhar
        self.indices = np.arange(len(self.tabela))
        self.on_epoch_end()

    def __len__(self) -> int:
        return int(
            np.ceil(
                len(self.tabela)
                / self.configuracao.tamanho_lote
            )
        )

    def __getitem__(self, indice_lote: int):
        inicio = indice_lote * self.configuracao.tamanho_lote
        fim = min(
            inicio + self.configuracao.tamanho_lote,
            len(self.tabela),
        )

        indices_lote = self.indices[inicio:fim]

        videos = []
        rotulos = []

        for indice in indices_lote:
            linha = self.tabela.iloc[indice]

            video = extrair_quadros(
                linha["caminho"],
                self.configuracao.quantidade_quadros,
                self.configuracao.tamanho_imagem,
            )

            if self.aumentar:
                video = aumentar_video(video)

            videos.append(video)
            rotulos.append(int(linha["rotulo"]))

        return (
            np.asarray(videos, dtype=np.float32),
            np.asarray(rotulos, dtype=np.float32),
        )

    def on_epoch_end(self):
        if self.embaralhar:
            np.random.shuffle(self.indices)

## 6. Criar o modelo

In [ ]:
def criar_modelo(configuracao: Configuracao) -> keras.Model:
    formato_video = (
        configuracao.quantidade_quadros,
        configuracao.tamanho_imagem,
        configuracao.tamanho_imagem,
        3,
    )

    entrada = keras.Input(
        shape=formato_video,
        name="video",
    )

    x = layers.Rescaling(
        1.0 / 127.5,
        offset=-1.0,
        name="normalizacao",
    )(entrada)

    extrator = keras.applications.MobileNetV2(
        input_shape=(
            configuracao.tamanho_imagem,
            configuracao.tamanho_imagem,
            3,
        ),
        alpha=configuracao.alfa_mobilenet,
        include_top=False,
        weights="imagenet",
        pooling="avg",
    )

    extrator.trainable = False

    x = layers.TimeDistributed(
        extrator,
        name="extrator_quadros",
    )(x)

    media_temporal = layers.GlobalAveragePooling1D(
        name="media_temporal",
    )(x)

    maximo_temporal = layers.GlobalMaxPooling1D(
        name="maximo_temporal",
    )(x)

    x = layers.Concatenate(
        name="resumo_temporal",
    )([media_temporal, maximo_temporal])

    x = layers.Dropout(
        configuracao.dropout,
        name="dropout",
    )(x)

    saida = layers.Dense(
        1,
        activation="sigmoid",
        dtype="float32",
        name="probabilidade_violencia",
    )(x)

    return keras.Model(
        entrada,
        saida,
        name="detector_violencia",
    )


def compilar_modelo(
    modelo: keras.Model,
    taxa_aprendizado: float,
):
    modelo.compile(
        optimizer=keras.optimizers.Adam(
            taxa_aprendizado
        ),
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.BinaryAccuracy(
                name="acuracia"
            ),
            keras.metrics.Precision(
                name="precisao"
            ),
            keras.metrics.Recall(
                name="revocacao"
            ),
            keras.metrics.AUC(
                name="auc"
            ),
        ],
    )


modelo = criar_modelo(configuracao)
compilar_modelo(
    modelo,
    configuracao.taxa_aprendizado,
)

modelo.summary()

## 7. Treinar a cabeça classificadora

In [ ]:
gerador_treino = GeradorVideos(
    treino,
    configuracao,
    aumentar=True,
    embaralhar=True,
)

gerador_validacao = GeradorVideos(
    validacao,
    configuracao,
    aumentar=False,
    embaralhar=False,
)

caminho_melhor_modelo = (
    PASTA_MODELOS / "melhor_modelo.keras"
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        caminho_melhor_modelo,
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=configuracao.paciencia,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1,
    ),
    keras.callbacks.CSVLogger(
        str(PASTA_RESULTADOS / "historico_treinamento.csv"),
        append=True,
    ),
]

historico_cabeca = modelo.fit(
    gerador_treino,
    validation_data=gerador_validacao,
    epochs=configuracao.epocas_cabeca,
    callbacks=callbacks,
    verbose=1,
)

## 7.1 Métricas da primeira fase



In [ ]:
metricas_cabeca = pd.DataFrame(
    historico_cabeca.history
)

display(metricas_cabeca)

metricas_cabeca.plot(
    figsize=(12, 6),
    grid=True,
    title="Métricas da primeira fase",
)

plt.xlabel("Época")
plt.show()

plt.figure(figsize=(8, 5))

plt.plot(
    historico_cabeca.history["loss"],
    label="treino",
)

plt.plot(
    historico_cabeca.history["val_loss"],
    label="validação",
)

plt.xlabel("Época")
plt.ylabel("Perda")
plt.title("Perda — primeira fase")
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(8, 5))

plt.plot(
    historico_cabeca.history["auc"],
    label="treino",
)

plt.plot(
    historico_cabeca.history["val_auc"],
    label="validação",
)

plt.xlabel("Época")
plt.ylabel("AUC")
plt.title("AUC — primeira fase")
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(8, 5))

plt.plot(
    historico_cabeca.history["revocacao"],
    label="treino",
)

plt.plot(
    historico_cabeca.history["val_revocacao"],
    label="validação",
)

plt.xlabel("Época")
plt.ylabel("Revocação")
plt.title("Revocação — primeira fase")
plt.legend()
plt.grid()
plt.show()

## 8. Ajuste fino

In [ ]:
extrator = modelo.get_layer("extrator_quadros").layer
extrator.trainable = True

for camada in extrator.layers:
    camada.trainable = False

for camada in extrator.layers[
    -configuracao.camadas_ajustaveis:
]:
    if not isinstance(
        camada,
        layers.BatchNormalization,
    ):
        camada.trainable = True

compilar_modelo(
    modelo,
    configuracao.taxa_ajuste,
)

epoca_inicial = len(historico_cabeca.epoch)

historico_ajuste = modelo.fit(
    gerador_treino,
    validation_data=gerador_validacao,
    initial_epoch=epoca_inicial,
    epochs=(
        epoca_inicial
        + configuracao.epocas_ajuste
    ),
    callbacks=callbacks,
    verbose=1,
)

modelo = keras.models.load_model(
    caminho_melhor_modelo
)

## 8.1 Métricas do ajuste fino


In [ ]:
metricas_ajuste = pd.DataFrame(
    historico_ajuste.history
)

display(metricas_ajuste)

metricas_ajuste.plot(
    figsize=(12, 6),
    grid=True,
    title="Métricas do ajuste fino",
)

plt.xlabel("Época")
plt.show()

plt.figure(figsize=(8, 5))

plt.plot(
    historico_ajuste.history["loss"],
    label="treino",
)

plt.plot(
    historico_ajuste.history["val_loss"],
    label="validação",
)

plt.xlabel("Época")
plt.ylabel("Perda")
plt.title("Perda — ajuste fino")
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(8, 5))

plt.plot(
    historico_ajuste.history["auc"],
    label="treino",
)

plt.plot(
    historico_ajuste.history["val_auc"],
    label="validação",
)

plt.xlabel("Época")
plt.ylabel("AUC")
plt.title("AUC — ajuste fino")
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(8, 5))

plt.plot(
    historico_ajuste.history["revocacao"],
    label="treino",
)

plt.plot(
    historico_ajuste.history["val_revocacao"],
    label="validação",
)

plt.xlabel("Época")
plt.ylabel("Revocação")
plt.title("Revocação — ajuste fino")
plt.legend()
plt.grid()
plt.show()

## 8.2 Histórico completo em CSV


In [ ]:
caminho_historico = (
    PASTA_RESULTADOS
    / "historico_treinamento.csv"
)

historico_csv = pd.read_csv(
    caminho_historico
)

display(historico_csv)

## 9. Avaliar no conjunto de teste

In [ ]:
gerador_teste = GeradorVideos(
    teste,
    configuracao,
    aumentar=False,
    embaralhar=False,
)

probabilidades = modelo.predict(
    gerador_teste,
    verbose=1,
).reshape(-1)

rotulos_reais = teste["rotulo"].to_numpy(dtype=int)

rotulos_previstos = (
    probabilidades >= configuracao.limiar
).astype(int)

relatorio = classification_report(
    rotulos_reais,
    rotulos_previstos,
    target_names=[
        "nao_violencia",
        "violencia",
    ],
    zero_division=0,
)

print(relatorio)

matriz = confusion_matrix(
    rotulos_reais,
    rotulos_previstos,
)

ConfusionMatrixDisplay(
    matriz,
    display_labels=[
        "nao_violencia",
        "violencia",
    ],
).plot()

plt.title("Matriz de confusão")
plt.show()

## 10. Curvas do treinamento

In [ ]:
historico = {}

for chave in set(
    historico_cabeca.history
) | set(
    historico_ajuste.history
):
    historico[chave] = (
        historico_cabeca.history.get(chave, [])
        + historico_ajuste.history.get(chave, [])
    )

plt.figure(figsize=(8, 5))
plt.plot(
    historico.get("loss", []),
    label="treino",
)
plt.plot(
    historico.get("val_loss", []),
    label="validação",
)
plt.xlabel("Época")
plt.ylabel("Perda")
plt.title("Perda durante o treinamento")
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(
    historico.get("auc", []),
    label="treino",
)
plt.plot(
    historico.get("val_auc", []),
    label="validação",
)
plt.xlabel("Época")
plt.ylabel("AUC")
plt.title("AUC durante o treinamento")
plt.legend()
plt.show()

## 11. Exportar para TensorFlow Lite

In [ ]:
caminho_float32 = (
    PASTA_MODELOS
    / "violencia_float32.tflite"
)

conversor_float32 = (
    tf.lite.TFLiteConverter
    .from_keras_model(modelo)
)

modelo_float32 = conversor_float32.convert()
caminho_float32.write_bytes(modelo_float32)

caminho_float16 = (
    PASTA_MODELOS
    / "violencia_float16.tflite"
)

conversor_float16 = (
    tf.lite.TFLiteConverter
    .from_keras_model(modelo)
)

conversor_float16.optimizations = [
    tf.lite.Optimize.DEFAULT
]

conversor_float16.target_spec.supported_types = [
    tf.float16
]

modelo_float16 = conversor_float16.convert()
caminho_float16.write_bytes(modelo_float16)

print("Float32:", caminho_float32)
print("Float16:", caminho_float16)

## 12. Salvar resultados

In [ ]:
resultado = {
    "configuracao": configuracao.__dict__,
    "quantidade_videos": {
        "treino": len(treino),
        "validacao": len(validacao),
        "teste": len(teste),
    },
    "matriz_confusao": matriz.tolist(),
    "relatorio": classification_report(
        rotulos_reais,
        rotulos_previstos,
        target_names=[
            "nao_violencia",
            "violencia",
        ],
        output_dict=True,
        zero_division=0,
    ),
}

with (
    PASTA_RESULTADOS
    / "avaliacao_teste.json"
).open(
    "w",
    encoding="utf-8",
) as arquivo:
    json.dump(
        resultado,
        arquivo,
        ensure_ascii=False,
        indent=2,
    )

predicoes = teste.copy()
predicoes["probabilidade_violencia"] = probabilidades
predicoes["predicao"] = rotulos_previstos

predicoes.to_csv(
    PASTA_RESULTADOS
    / "predicoes_teste.csv",
    index=False,
)

print("Resultados salvos em:", PASTA_RAIZ)

## 13. Compactar e baixar

In [ ]:
from google.colab import files
import shutil

arquivo_zip = shutil.make_archive(
    "/content/resultados_violencia",
    "zip",
    root_dir=str(PASTA_RAIZ),
)

print("Arquivo gerado:", arquivo_zip)
files.download(arquivo_zip)